## Parte 5: Machine Learning

## Parte 5: Machine Learning Geoespacial
### Problema:
La planificación urbana y la gestión territorial requieren comprender cómo se distribuye espacialmente la densidad urbana, entendida como la concentración de edificaciones en el territorio. Esta información es fundamental para orientar decisiones sobre infraestructura, movilidad, provisión de servicios y procesos de expansión urbana. Sin embargo, los registros detallados sobre la cantidad de edificaciones por unidad espacial suelen ser incompletos, poco actualizados o no integrados con otras variables territoriales.

La densidad urbana surge de interacciones espaciales complejas y no lineales, influenciadas por factores geométricos, proximidad, estructura del entorno construido y características del territorio, lo que dificulta su estimación mediante enfoques tradicionales. Además, los patrones urbanos presentan autocorrelación espacial, lo que exige modelos capaces de generalizar hacia zonas sin datos directos.

En este contexto, el problema consiste en modelar y predecir la distribución espacial de la variable edificios_count, que representa el número de edificaciones por unidad territorial, utilizando variables espaciales y territoriales disponibles. El objetivo es evaluar la capacidad de modelos de aprendizaje automático (random forest, xgboost y svm) para capturar patrones urbanos reales bajo un esquema de validación espacial. El desafío es determinar si es posible reconstruir la densidad urbana de manera consistente y coherente en ausencia de información directa en determinadas zonas, y establecer qué enfoques de modelamiento resultan más adecuados para este fenómeno.

#### Tipo de problema: Regresión espacial
#### Unidad espacial: Manzana urbana

In [1]:
# 1. Setup y carga librerías
import os, pandas as pd, geopandas as gpd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv
from shapely.geometry import Point
load_dotenv()
sns.set_palette('viridis')
POSTGRES_USER=os.getenv('POSTGRES_USER'); POSTGRES_PASSWORD=os.getenv('POSTGRES_PASSWORD')
POSTGRES_HOST=os.getenv('POSTGRES_HOST','localhost'); POSTGRES_PORT=os.getenv('POSTGRES_PORT','5432'); POSTGRES_DB=os.getenv('POSTGRES_DB')
engine = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}')
print('Conexión lista')

Conexión lista


In [3]:
# 2. Carga de datasets clave
from pathlib import Path
processed_dir = Path('../data/processed')

def load_gdf(schema, table):
    try:
        return gpd.read_postgis(f'SELECT * FROM {schema}.{table}', engine, geom_col='geometry')
    except Exception as e:
        print(f'Error cargando {schema}.{table}:', e)
        return None
manz_attr = load_gdf('processed_data','manzanas_atributos')
uso = load_gdf('processed_data','manzanas_uso_suelo')
net_nodes = load_gdf('processed_data','network_nodes_metrics')
metrics = pd.read_sql('SELECT * FROM processed_data.metrics_manzanas', engine)
print('Shapes:', len(manz_attr) if manz_attr is not None else None, len(metrics))
required_files = ['sentinel2_ndvi.tif','sentinel2_ndvi_32719.tif','metrics_manzanas.csv']
for f in required_files:
    exists = (processed_dir / f).exists()
    print(f'Check {f}:', 'OK' if exists else 'FALTA')
# Se validará lisa_clusters.geojson tras su export en celda 8D

Shapes: 792 792
Check sentinel2_ndvi.tif: OK
Check sentinel2_ndvi_32719.tif: FALTA
Check metrics_manzanas.csv: OK


In [ ]:
# 3. Unificación de atributos clave por manzana (+ densidad vial derivada)
# Selección mínima de columnas relevantes para exploración
merged = metrics.copy()
# Asegurar alias clave
key = 'manzent' if 'manzent' in merged.columns else ('MANZENT' if 'MANZENT' in merged.columns else merged.columns[0])
if manz_attr is not None:
    geo_subset = manz_attr[[key, 'geometry']].copy()
    geo_subset.loc[:, key] = geo_subset[key].astype(str)
    merged.loc[:, key] = merged[key].astype(str)
    merged_geo = geo_subset.merge(merged, on=key, how='left')
else:
    merged_geo = None
# Densidad vial derivada si faltante
if merged_geo is not None:
    if 'road_density_m_per_km2' not in merged_geo.columns and 'road_length_m' in merged_geo.columns:
        # Buscar área en metros cuadrados
        area_col = None
        for cand in ['area_m2','Shape__Area','shape_area','AREA']:
            if cand in merged_geo.columns:
                area_col = cand
                break
        if area_col is not None:
            merged_geo['road_density_m_per_km2'] = merged_geo.apply(lambda r: r['road_length_m'] / (r[area_col]/1e6) if r[area_col] and r[area_col] > 0 else np.nan, axis=1)
            print('Creada road_density_m_per_km2 a partir de road_length_m y', area_col)
        else:
            print('No se encontró columna de área para derivar densidad vial.')
numeric_cols = [c for c in merged.columns if merged[c].dtype != 'object' and c not in ['geometry']]
print('Columnas merged:', merged.columns.tolist()[:15])

In [ ]:
from shapely.geometry import Point
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_score
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import shap

TARGET = 'edificios_count'
print("Variable objetivo:", TARGET)
print("Disponible en columnas:", TARGET in merged_geo.columns)

# Asegurar CRS métrico
gdf = merged_geo.copy()
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=32719)
elif gdf.crs.to_epsg() != 32719:
    gdf = gdf.to_crs(epsg=32719)


In [ ]:
## Feature engineering espacial

gdf = merged_geo.copy()

# DataFrame de features
features = pd.DataFrame(index=gdf.index)

# Coordenadas (centroides)
centroids = gdf.geometry.centroid
features['x'] = centroids.x
features['y'] = centroids.y

# Área de la manzana
for col in ['area_m2','Shape__Area','shape_area','AREA']:
    if col in gdf.columns:
        features['area_m2'] = gdf[col]
        break

# Densidad local (buffers)-
for radius in [300, 600, 1000]:
    buffer = centroids.buffer(radius)
    features[f'density_{radius}m'] = buffer.apply(
        lambda b: centroids.within(b).sum()
    )

# Variables numéricas existentes
numeric_cols = gdf.select_dtypes(include=np.number).columns.tolist()

# Eliminar:
# - variable objetivo
# - variables ya creadas
# - otras densidades para evitar leakage
# - IDs puros
numeric_cols = [
    c for c in numeric_cols
    if c != TARGET
    and not c.startswith('density_')
    and c not in features.columns
    and not c.lower().startswith('id')
]

features = pd.concat(
    [features, gdf[numeric_cols]],
    axis=1
)

print("Número total de features:", features.shape[1])

from sklearn.cluster import KMeans
# Validación espacial (GroupKFold)
# Usamos coordenadas espaciales
coords = X_clean[['x', 'y']].values

# Número de zonas espaciales (ideal entre 8 y 20)
N_ZONES = 10

kmeans = KMeans(
    n_clusters=N_ZONES,
    random_state=42,
    n_init=10
)

groups_clean = kmeans.fit_predict(coords)

print("Zonas espaciales:", np.unique(groups_clean).size)

# Plot de las zonas espaciales
gdf.loc[X_clean.index, 'spatial_zone'] = groups_clean

fig, ax = plt.subplots(1,1, figsize=(8,8))
gdf.plot(
    column='spatial_zone',
    categorical=True,
    legend=True,
    ax=ax
)
ax.set_title('Zonas espaciales para validación (KMeans)')
ax.axis('off')
plt.show()



In [ ]:
# Variables explicativas existentes
X = features.copy()
y = gdf[TARGET]

# Unir para limpieza coherente
ml_df = X.copy()
ml_df[TARGET] = y

# Eliminar filas sin target
ml_df = ml_df.dropna(subset=[TARGET])

# Imputación simple (mediana)
X_clean = ml_df.drop(columns=[TARGET]).fillna(
    ml_df.drop(columns=[TARGET]).median()
)
y_clean = ml_df[TARGET]

print("Registros finales:", len(X_clean))
print("NaN en X:", X_clean.isna().sum().sum())
print("NaN en y:", y_clean.isna().sum())

In [ ]:
## Comparación de modelos

# random forest
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

cv = GroupKFold(n_splits=5)

rf_scores = cross_val_score(
    rf,
    X_clean,
    y_clean,
    cv=cv,
    groups=groups_clean,
    scoring='r2'
)

print(f"RF R2 (Spatial CV): {rf_scores.mean():.3f} ± {rf_scores.std():.3f}")


In [ ]:
# XGBoost
xgb = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb_scores = cross_val_score(
    xgb,
    X_clean,
    y_clean,
    cv=cv,
    groups=groups_clean,
    scoring='r2'
)

print(f"XGB R2 (Spatial CV): {xgb_scores.mean():.3f} ± {xgb_scores.std():.3f}")



In [ ]:
# SVM espacial
svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(C=10, gamma='scale'))
])

svm_scores = cross_val_score(
    svm,
    X_clean,
    y_clean,
    cv=cv,
    groups=groups_clean,
    scoring='r2'
)

print(f"SVM R2 (Spatial CV): {svm_scores.mean():.3f} ± {svm_scores.std():.3f}")



In [ ]:
## Entrenamiento final + predicción espacial
rf.fit(X_clean, y_clean)

# Predicción para todas las manzanas
X_all = features.fillna(X_clean.median())

pred = rf.predict(X_all)
gdf['prediction'] = pred

gdf[['prediction']].head()

In [ ]:
## Mapas de Predicción e Incertidumbre

# prediccion
fig, ax = plt.subplots(1,1, figsize=(8,8))
gdf.plot(column='prediction', cmap='viridis', legend=True, ax=ax)
ax.set_title('Predicción espacial de densidad urbana (RF)')
ax.axis('off')
plt.show()

# incertidumbre
tree_preds = np.array([
    tree.predict(X_all) for tree in rf.estimators_
])

gdf['uncertainty'] = tree_preds.std(axis=0)

fig, ax = plt.subplots(1,1, figsize=(8,8))
gdf.plot(column='uncertainty', cmap='inferno', legend=True, ax=ax)
ax.set_title('Incertidumbre espacial del modelo')
ax.axis('off')
plt.show()

## Interpretación del modelo

# Features importantes
importances = pd.Series(
    rf.feature_importances_,
    index=X_clean.columns
).sort_values(ascending=False)

importances.head(15).plot(
    kind='barh', figsize=(6,6)
)
plt.title('Importancia de variables (RF)')
plt.gca().invert_yaxis()
plt.show()

# SHAP Values
explainer = shap.TreeExplainer(rf)

sample_idx = np.random.choice(
    X_clean.index, size=min(500, len(X_clean)), replace=False
)

shap_values = explainer.shap_values(X_clean.loc[sample_idx])

shap.summary_plot(
    shap_values,
    X_clean.loc[sample_idx]
)


## ✔️ Checklist de Avances — Machine Learning Geoespacial

### 1.Definición del problema
### 2. Feature engineering espacial completo
### 3. MRF, XGBoost, SV
### 4. Validación espacial (GroupKFold)
### 5. Mapas de predicción
### 6. Incertidumbre
### 7. Interpretación (SHAP + importance)
